# Market Regime Allocation with Hidden Markov Models

## 1. Project Motivation

Markets do not behave the same way all the time.

Sometimes the market is calm: equities trend upward, volatility is low, and investors are willing to take risk. At other times, the market becomes stressed: prices fall quickly, volatility rises, and assets that usually diversify a portfolio may stop behaving as expected.

This project is about building a systematic way to identify those changing market conditions and test whether they can support better portfolio allocation decisions.

### Simple intuition

A simple way to think about market regimes is to compare markets to weather.

If the weather is sunny, you dress differently than if there is a storm. You do not need to know the exact weather tomorrow to understand that an umbrella is useful when storm conditions are becoming more likely.

Market regimes are similar. The goal is not to predict every daily market move. The goal is to estimate the current environment, such as calm growth, high-volatility stress, or inflation/rate pressure, and then ask whether the portfolio should be positioned differently under those conditions.

### Why regimes matter

Many investment strategies implicitly assume that market relationships are reasonably stable. For example, a balanced portfolio often assumes that equities provide growth while bonds help reduce risk during equity selloffs.

In reality, these relationships can change. During some stress periods, volatility rises, risky assets fall together, and diversification becomes less reliable. During inflation or rate-hiking environments, long-duration bonds may not hedge equities as well as they do during growth shocks.

A good example is 2022. Many investors had become used to the idea that long-term Treasury bonds could help hedge equity drawdowns. This worked well in periods such as the dot-com crash and the 2008 Global Financial Crisis, where recession risk led central banks to cut interest rates. Falling interest rates usually push bond prices up, which can help offset equity losses.

In 2022, the situation was different. Inflation was high, and the Federal Reserve raised interest rates aggressively to bring inflation under control. Equities fell, but long-duration Treasury bonds also fell because yields were rising. The usual stock-bond hedge failed because the market environment was different: the dominant risk was not just recession risk, but inflation and rate pressure.

This matters because a portfolio that is appropriate in one environment may become too risky in another. A regime-aware process tries to recognize when the environment has changed instead of treating all historical periods as if they came from the same market condition.

### Research background

The idea of regime shifts is well established in economics and finance.

- Hamilton's classic Markov-switching model introduced a way to model economic time series as moving between hidden states, such as expansion and recession-like conditions. The important idea for this project is that these states are not directly observed. They are inferred from data as probabilities, not treated as objective labels. See Hamilton (1989), ["A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle"](https://www.econometricsociety.org/publications/econometrica/1989/03/01/new-approach-economic-analysis-nonstationary-time-series-and).
- Ang and Bekaert studied time-varying correlations and found evidence that volatility and correlations can behave differently across regimes. This supports the idea that risk itself can be state-dependent, rather than constant through time. See their NBER working paper, ["International Asset Allocation with Time-Varying Correlations"](https://www.nber.org/papers/w7056).
- Ang and Bekaert's ["How Do Regimes Affect Asset Allocation?"](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=310626) is especially relevant to this project because it connects regime shifts directly to portfolio allocation. The key motivation is not only to identify regimes, but to ask whether different regimes should lead to different portfolio choices.
- In practical portfolio management, regime thinking is useful because risk is not only about average return. It is also about drawdowns, volatility, correlation breakdowns, and whether the portfolio is exposed to the wrong risks at the wrong time.

This project applies that broad idea to an ETF allocation setting: use market and macroeconomic signals to estimate hidden market conditions, then test whether those estimated conditions can guide risk-aware allocation decisions.

### Project question

The central question is:

> Can market and macroeconomic signals be used to identify probabilistic market regimes, and can those regimes support a systematic ETF allocation strategy that improves risk-adjusted performance and drawdown control compared to simple static benchmarks?

In simpler words:

> Can we build a model that recognizes different market conditions, then use those conditions to decide when the portfolio should be more aggressive, more defensive, or more diversified?

### What this project is not trying to do

This project is not trying to prove that a machine learning model can perfectly predict the market.

It is also not trying to build a black-box trading system that automatically maximizes returns.

Instead, this project treats machine learning as a risk analysis tool. The Hidden Markov Model estimates the likely market regime. The strategy then uses transparent allocation rules for each regime. This distinction is important: the model helps describe the environment, while the allocation rule converts that environment into a portfolio decision.

### What success looks like

A useful result does not have to mean beating the stock market on raw return.

Because this is a risk-aware allocation project, success will be judged using metrics such as:

- lower maximum drawdown,
- lower volatility,
- better Sharpe or Sortino ratio,
- better behavior during stress periods,
- reasonable turnover after transaction costs,
- clear and economically interpretable regimes.

A strong project should also be honest if the model does not add value beyond simple benchmarks. In that case, the learning is still useful: it tells us that a more complex regime model must justify itself against simpler allocation rules.

## 2. Asset Universe and Data Sources

The asset universe matters because the model does not trade in isolation. Even if the regime model identifies a useful market condition, the strategy can only respond through the assets available in the portfolio.

For this project, the initial allocation universe is built around four liquid ETFs: SPY, TLT, GLD, and SHY. These ETFs are chosen because they represent different portfolio roles rather than four versions of the same risk exposure.

### Why these assets?

| ETF | Asset class | Portfolio role |
|---|---|---|
| SPY | U.S. equities | Growth and risk-on exposure |
| TLT | Long-term U.S. Treasuries | Duration exposure and potential recession hedge |
| GLD | Gold | Inflation, uncertainty, and alternative defensive exposure |
| SHY | Short-term U.S. Treasuries | Defensive cash-like exposure |

SPY represents broad U.S. equity market exposure. It is the main growth asset in the portfolio and is usually expected to do well when investors are willing to take risk.

TLT represents long-duration U.S. Treasury bonds. These bonds can perform well when growth slows and interest rates fall, but they can suffer when inflation and rising rates are the main concern.

GLD represents gold exposure. Gold does not behave like a stock or bond, so it can be useful when the market is worried about inflation, currency risk, or broader uncertainty.

SHY represents short-term U.S. Treasury exposure. It is included as a lower-volatility defensive asset, similar to a place where the strategy can reduce risk when the environment is unfavorable or unclear.

### Link to regime-based allocation

This asset universe connects directly to the regime allocation idea discussed earlier.

If regimes affect returns, volatility, and correlations, then the same portfolio may not be suitable in every environment. A calm growth regime may justify more equity exposure. A recession-like stress regime may favor Treasuries or short-duration defensive assets. An inflation or rate-pressure regime may require more caution toward long-duration bonds and may make gold or short-term Treasuries more useful.

This is why the project uses assets with different economic roles. The point is not just to identify regimes, but to test whether those regime estimates can lead to more sensible allocation choices.

### Data sources

This project uses two main data sources:

| Source | What it provides | How it is used |
|---|---|---|
| Tiingo | Historical ETF price data | Asset returns and market-based features |
| FRED | Macroeconomic and risk indicators | Macro context and regime features |

Tiingo provides the historical ETF prices needed to calculate portfolio returns. From these prices, we can engineer market features such as returns, momentum, rolling volatility, drawdowns, and correlations.

FRED provides macroeconomic and risk-related time series, such as interest rates, yield spreads, inflation measures, credit spreads, and recession indicators. These features help the model distinguish between market environments that may look similar from prices alone but are driven by different economic forces.

### Why these data sources are sufficient

The strategy is based on monthly allocation decisions, not high-frequency trading. Because of that, the project does not require intraday prices, order book data, or extremely detailed execution data.

Tiingo is sufficient for the market data side because the key raw input is adjusted historical ETF prices. Indicators such as volatility, momentum, trend, drawdown, and correlation do not need to come directly from the data provider. They can be calculated from the price history, which makes the feature definitions transparent and easier to check for lookahead bias.

FRED is sufficient for the macro side because it provides widely used public economic data. The main challenge is not finding the most complicated macro dataset, but using macro data carefully. In particular, macro features should be lagged so that the backtest does not accidentally use information that would not have been available at the time.

In short: Tiingo gives the market prices, FRED gives the economic context, and the project turns both into interpretable regime features.